# 04 - Paired headline generation: pilot

**What this does.** Takes 200 headlines from the **training split** (half sarcastic, half not) and asks three open LLMs (Qwen, Gemma, Mistral) to write, for each one, a **sarcastic** and a **serious** headline on the same topic. Each seed goes to one model; both sides of a pair come from that model. Then it checks the outputs automatically, asks a fourth open model from a different family (Microsoft Phi) to judge every headline blind (it also judges the 200 real seeds, to measure how good the judge is), and writes a blind sheet for your own check.

**Why a pilot.** Before generating thousands of pairs we need to know: does each model follow the format, how often does it refuse, does the sarcasm come out sarcastic, and how long does it take. Nothing here is used for training yet.

**Before you run, in the right-hand panel:**
- **Accelerator: GPU T4 x2.** Each model is split across both GPUs.
- **Internet: on.** Code from GitHub, models from Hugging Face.
- **Input:** your private dataset **`gainsplit-processed`** (any version; only `headlines_train.csv` and `cue_words.json` are read).
- **Secret** (Add-ons > Secrets), attached to this notebook: **`HF_TOKEN`** (needed for Gemma only). Never paste a token into a cell.

**Time and cost.** Free (Kaggle GPU only). Loading each generator takes a few minutes (about 15-18 GB download each), the judge about 8 GB; generation time is what this pilot measures.

**Output:** `/kaggle/working/pairgen_pilot/`, zipped in the last step. In an interactive session, download the zip before closing.

## 1. Get the code
Same as the training notebook: clones the public repo and prints the commit, then runs the tests for the pair-generation helpers (no GPU needed, a few seconds).

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")

COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_pairgen.py", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 2. Settings
- `MODELS`: short name -> Hugging Face model id. Swap an id here to try a newer version of a family.
- `GEN`: sampling settings, the same for every model. Some randomness (`temperature` 0.8) gives varied jokes; the `seed` makes the run repeatable on the same hardware.
- Gemma 2 is meant to run in bfloat16, which T4 GPUs do not support well, so here it runs in float16 with the "eager" attention it recommends. If its outputs come out empty or garbled, that is the reason; report it and we change the setup.

In [ ]:
from pathlib import Path

N_SEEDS = 200          # half sarcastic, half not
SEED = 2026            # seed choice, model assignment and sampling

MODELS = {
    "qwen":    {"id": "Qwen/Qwen2.5-7B-Instruct",           "extra": {}},
    "gemma":   {"id": "google/gemma-2-9b-it",               "extra": {"attn_implementation": "eager"}},
    "mistral": {"id": "mistralai/Mistral-7B-Instruct-v0.3", "extra": {}},
}
GEN = {"max_new_tokens": 120, "temperature": 0.8, "top_p": 0.95, "batch_size": 16}

JUDGE_MODEL = "microsoft/Phi-3.5-mini-instruct"   # a family not used for generation; not gated
JUDGE_BATCH = 32
TRAIN_SEEDS_TOTAL = 22800   # size of the training split, for the time estimate at the end

OUT = Path("/kaggle/working/pairgen_pilot")
OUT.mkdir(parents=True, exist_ok=True)

## 3. Data: pick the seed headlines
Finds `gainsplit-processed`, reads **only the training split**, and draws 200 seeds (100 per class). Each seed is assigned to one model, in equal shares and random order. Dev and test headlines are never read, so no test topic can leak into generated training data.

In [ ]:
import json
import pandas as pd
from gainsplit.data import find_data_dir
from gainsplit import pairgen as pg

DATA_DIR = find_data_dir("/kaggle/input")
train = pd.read_csv(DATA_DIR / "headlines_train.csv", keep_default_na=False)
print("data:", DATA_DIR, "| train rows:", len(train))

seeds = pg.pick_seeds(train, N_SEEDS, SEED)
seeds["model"] = pg.assign_models(len(seeds), list(MODELS), SEED)
assert seeds["seed_id"].isin(train["id"]).all()
seeds.to_csv(OUT / "seeds.csv", index=False)

print(pd.crosstab(seeds["model"], seeds["seed_label"], margins=True))
seeds.head()

## 4. Secrets and GPUs
Reads the Hugging Face token from Kaggle Secrets if it is attached (Qwen and Mistral work without it; Gemma needs it).

In [ ]:
import torch

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

HF_TOKEN = get_secret("HF_TOKEN")
print("HF_TOKEN found:", HF_TOKEN is not None)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 5. Generate
For each model: load it across both GPUs, write the pairs for its seeds in batches, save the raw output, then free the GPUs before the next model. Raw output is saved per model (`raw_<model>.csv`), so if the session stops you can run this cell again and finished models are skipped.

The prompt (the same for every model) is printed first so you can read exactly what the models were asked.

In [ ]:
import gc, time
from transformers import AutoTokenizer, AutoModelForCausalLM

print(pg.build_prompt("<seed headline goes here>"))

def load(model_id, extra):
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    tok.padding_side = "left"                     # pad on the left so generation continues right after the prompt
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, token=HF_TOKEN, torch_dtype=torch.float16, device_map="auto", **extra)
    model.eval()
    return tok, model

def generate(tok, model, prompts):
    torch.manual_seed(SEED)
    outs = []
    for i in range(0, len(prompts), GEN["batch_size"]):
        chunk = prompts[i:i + GEN["batch_size"]]
        texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
                 for p in chunk]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=GEN["max_new_tokens"], do_sample=True,
                                 temperature=GEN["temperature"], top_p=GEN["top_p"], pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

timing = json.loads((OUT / "timing.json").read_text()) if (OUT / "timing.json").exists() else {}
for name, spec in MODELS.items():
    path = OUT / f"raw_{name}.csv"
    if path.exists():
        print(name, "already done, skipped"); continue
    part = seeds[seeds["model"] == name].reset_index(drop=True)
    t0 = time.time()
    tok, model = load(spec["id"], spec["extra"])
    t1 = time.time()
    part["raw"] = generate(tok, model, [pg.build_prompt(t) for t in part["seed_text"]])
    t2 = time.time()
    part.to_csv(path, index=False)
    timing[name] = {"model_id": spec["id"], "load_s": round(t1 - t0, 1), "gen_s": round(t2 - t1, 1),
                    "seeds": len(part), "gen_s_per_seed": round((t2 - t1) / len(part), 3)}
    (OUT / "timing.json").write_text(json.dumps(timing, indent=2))
    print(name, timing[name])
    del model, tok; gc.collect(); torch.cuda.empty_cache()

pd.DataFrame(timing).T

## 6. Automatic checks
For each output: did the model refuse, is there valid JSON with both headlines, is either side empty or longer than 30 words, are the two sides identical, does either side just copy the seed. A pair is **ok** only if all checks pass. Text is cleaned the same way as the processed data (lowercase, punctuation removed).

The last table is the **outlet-cue check**: the share of headlines containing one of the 100 words that most strongly point to each class in the real training data (`cue_words.json`, Pass 3). If the generated sarcastic side is full of "nation", "area" and the like while the serious side is not, the models are copying The Onion's house style and the shortcut comes back.

In [ ]:
raw = pd.concat([pd.read_csv(OUT / f"raw_{m}.csv", keep_default_na=False) for m in MODELS], ignore_index=True)
checks = pd.DataFrame([pg.check_pair(s, r) for s, r in zip(raw["seed_text"], raw["raw"])])
pairs = pd.concat([raw, checks], axis=1)
pairs.to_csv(OUT / "pairs_checked.csv", index=False)

flags = ["refusal", "parse_ok", "empty", "too_long", "same_pair", "copies_seed", "ok"]
display(pairs.groupby("model")[flags].mean().round(3))
display(pairs.groupby("model")[["words_sarcastic", "words_serious", "overlap_pair"]].mean().round(2))

cues = json.loads((DATA_DIR / "cue_words.json").read_text())
sarc_cues = [w for w, _ in cues["sarcastic"]]
serious_cues = [w for w, _ in cues["non_sarcastic"]]
good = pairs[pairs["ok"]]
rows = []
for name, g in good.groupby("model"):
    for side in ("sarcastic", "serious"):
        rows.append({"source": f"{name}: {side} side", "sarc_cue_rate": pg.cue_rate(g[side], sarc_cues),
                     "serious_cue_rate": pg.cue_rate(g[side], serious_cues)})
for lab, side in ((1, "sarcastic"), (0, "serious")):
    texts = seeds.loc[seeds["seed_label"] == lab, "seed_text"]
    rows.append({"source": f"real seeds: {side}", "sarc_cue_rate": pg.cue_rate(texts, sarc_cues),
                 "serious_cue_rate": pg.cue_rate(texts, serious_cues)})
display(pd.DataFrame(rows).round(3))

**Read a few pairs yourself** (5 per model, random). Does the sarcastic one read like satire? Is the serious one plain news? Do they share the topic?

In [ ]:
pd.set_option("display.max_colwidth", 120)
for name, g in pairs.groupby("model"):
    print(f"\n===== {name} =====")
    for p in g.sample(min(5, len(g)), random_state=SEED).itertuples():
        print(f"seed      : {p.seed_text}")
        print(f"sarcastic : {p.sarcastic if p.parse_ok else '(no parse) ' + str(p.raw)[:150]}")
        print(f"serious   : {p.serious}\n")

## 7. Judge (Phi, open model)
Every headline is shown **alone** (no pair, no hint of what was intended) and the judge answers SARCASTIC or SERIOUS. It also judges the **200 real seed headlines**, whose labels we know, so we can see how accurate the judge is on human-written headlines before trusting it on generated ones.

The judge is Microsoft Phi, a model family not used for generation (a model tends to approve of its own writing). It answers without randomness (greedy decoding), so the same headline always gets the same verdict. Answers it does not phrase as one of the two words are counted as unclear.

In [ ]:
tok, model = load(JUDGE_MODEL, {})

def judge_batch(texts):
    prompts = [tok.apply_chat_template([{"role": "user", "content": pg.judge_prompt(t)}], tokenize=False,
                                       add_generation_prompt=True) for t in texts]
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

items = pg.long_items(good)
real = seeds.rename(columns={"seed_text": "text", "seed_label": "intended"}).assign(
    item_id=lambda d: d["seed_id"].astype(str) + "_real", model="real")[["item_id", "seed_id", "model", "intended", "text"]]
to_judge = pd.concat([real, items], ignore_index=True)
print("headlines to judge:", len(to_judge))

t0 = time.time()
answers = []
for i in range(0, len(to_judge), JUDGE_BATCH):
    answers += judge_batch(list(to_judge["text"].iloc[i:i + JUDGE_BATCH]))
print(f"judging took {time.time() - t0:.0f} s")
del model, tok; gc.collect(); torch.cuda.empty_cache()

judged = to_judge.assign(judge_raw=[a.strip()[:40] for a in answers], judge=[pg.parse_judge(a) for a in answers])
judged["agrees"] = judged["judge"] == judged["intended"]
judged.to_csv(OUT / "judged.csv", index=False)
print("unclear answers:", int(judged["judge"].isna().sum()), "of", len(judged))

summary = (judged.assign(side=judged["intended"].map({1: "sarcastic", 0: "serious"}))
           .groupby(["model", "side"]).agg(n=("agrees", "size"), judge_agrees=("agrees", "mean"),
                                           unclear=("judge", lambda s: s.isna().mean())).round(3))
display(summary)

**How to read this.** The `real` rows are the judge's accuracy on human headlines: if it is far below about 0.85 there, the judge itself is weak and its verdicts on generated text mean little. For each model, `judge_agrees` on the sarcastic side is the share of its sarcastic headlines that actually read as sarcastic, and on the serious side the share that read as serious.

**Pairs kept**: a pair survives only if both sides pass the automatic checks and the judge agrees with both sides.

In [ ]:
gen_judged = judged[judged["model"] != "real"]
both_agree = gen_judged.groupby("seed_id")["agrees"].agg(lambda s: bool(len(s) == 2 and s.all()))
pairs["judge_both"] = pairs["seed_id"].map(both_agree).fillna(False).astype(bool)
pairs["kept"] = pairs["ok"] & pairs["judge_both"]
pairs.to_csv(OUT / "pairs_checked.csv", index=False)
kept = pairs.groupby("model").agg(seeds=("kept", "size"), ok=("ok", "mean"), kept=("kept", "mean")).round(3)
display(kept)
print("kept share by seed class (0 = serious seed, 1 = sarcastic seed):")
display(pairs.pivot_table(index="model", columns="seed_label", values="kept", aggfunc="mean").round(3))

## 8. Your blind check
Writes `audit_sheet.csv`: 100 random generated headlines with only a number and the text. Open it in a spreadsheet and fill `your_label` with **S** (sarcastic), **N** (serious) or **U** (unsure), without looking at `audit_key.csv`. Afterwards we compare your labels with what was intended and with the judge, which tells us how far to trust the judge for the full run.

In [ ]:
sheet, key = pg.blind_audit(items, n=100, seed=SEED)
sheet.to_csv(OUT / "audit_sheet.csv", index=False)
key.to_csv(OUT / "audit_key.csv", index=False)
print("audit_sheet.csv and audit_key.csv written:", len(sheet), "headlines")

## 9. Time estimate and summary
Generation seconds per seed for each model, and what the full training split (22,800 seeds, shared equally across the models) would take with this setup. Model loading time is added once per model.

In [ ]:
t = pd.DataFrame(timing).T
per_model = TRAIN_SEEDS_TOTAL / len(MODELS)
t["full_run_hours"] = ((t["gen_s_per_seed"].astype(float) * per_model + t["load_s"].astype(float)) / 3600).round(2)
display(t)
print("full split, all models one after another: about", round(t["full_run_hours"].sum(), 1), "GPU-hours")

result = {
    "commit": COMMIT, "n_seeds": N_SEEDS, "seed": SEED, "models": MODELS, "gen": GEN, "judge_model": JUDGE_MODEL,
    "timing": timing,
    "checks_ok_by_model": pairs.groupby("model")["ok"].mean().round(3).to_dict(),
    "kept_by_model": pairs.groupby("model")["kept"].mean().round(3).to_dict(),
    "judge_accuracy_on_real": round(float(judged.loc[judged["model"] == "real", "agrees"].mean()), 3),
    "judge_unclear": int(judged["judge"].isna().sum()),
}
(OUT / "summary.json").write_text(json.dumps(result, indent=2, default=str))
print(json.dumps(result, indent=2, default=str))

## 10. Download
Zips the output folder. Download `pairgen_pilot.zip` from the Output panel and keep it in `results/pairgen_pilot/` in your local repo. Then send the printed summary, the tables from steps 6-7 and anything that looked odd in the example pairs.

In [ ]:
import shutil
zip_path = shutil.make_archive("/kaggle/working/pairgen_pilot", "zip", OUT)
print(zip_path, "-", sorted(p.name for p in OUT.iterdir()))